In [2]:
import pandas as pd
import csv

# ========== 全局去重配置 ==========
DEDUP_KEEP = 'first'   # 'first' 或 'last'

# ========== 标签合并函数 ==========
def combine_labels_by_any(df, label_columns):
    """多列二值标签（0/1）：只要任意一列为1，结果即为1，否则0"""
    existing = [col for col in label_columns if col in df.columns]
    if not existing:
        return pd.Series([''] * len(df))
    combined = df[existing].max(axis=1).fillna(0).astype(int)
    return combined

# ========== 文件配置列表 ==========
file_configs = [
    # {
    #     "path": "es_hf_102024.csv",
    #     "sep": ",",
    #     "tweet_column": "text",
    #     "label_config": {
    #         "columns": [
    #             "Hate_Speech",
    #             "Targeted_Harassment",
    #             "NSFW_Sexual",
    #             "Violence_Incitement",
    #             "Dangerous_Ideology",
    #             "Profanity_Slang"
    #         ],
    #         "func": combine_labels_by_any   # 使用默认合并函数
    #     },
    #     "header": 0
    # },
    # 其他文件示例（单列标签）
    {
        "path": "es_hf_102024.csv",
        "sep": ",",
        "tweet_column": "text",
        "label_config": "labels",   # 直接写列名
        "header": 0
    },
    {
        "path": "es.csv",
        "sep": ",",
        "tweet_column": "text",
        "label_config": "label",   # 直接写列名
        "header": 0
    },
    # {
    #     "path": "in_hf.csv",
    #     "sep": ",",
    #     "tweet_column": "text",
    #     "label_config": "labels",   # 直接写列名
    #     "header": 0
    # },
    
]

# ========== 读取与提取函数 ==========
def read_file_with_fallback(file_config):
    file_path = file_config["path"]
    sep = file_config.get("sep", ",")
    header = file_config.get("header", 0)
    encoding_list = file_config.get("encoding_try", ['utf-8', 'utf-8-sig', 'latin-1', 'cp1252'])
    for enc in encoding_list:
        try:
            df = pd.read_csv(file_path, sep=sep, header=header, encoding=enc)
            print(f"  ✓ 使用编码 {enc} 成功读取 {file_path}")
            return df
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"  ✗ 使用编码 {enc} 读取失败: {e}")
            continue
    print(f"  ⚠ 所有编码均失败，使用 latin-1 强制读取")
    return pd.read_csv(file_path, sep=sep, header=header, encoding='latin-1')

def extract_tweets_and_labels(df, tweet_column, label_config):
    """提取推文和标签，支持单列字符串或多列字典配置"""
    if tweet_column not in df.columns:
        print(f"  ✗ 未找到推文列 '{tweet_column}'，实际列名: {df.columns.tolist()}")
        return [], []
    
    # 删除推文为空的记录
    non_empty = df[tweet_column].notna()
    valid_df = df[non_empty].copy()
    tweets = valid_df[tweet_column].tolist()

    # 处理标签
    if label_config is None:
        labels = [''] * len(tweets)
    elif isinstance(label_config, str):
        # 旧式：单一列名
        if label_config not in df.columns:
            print(f"  ⚠ 未找到标签列 '{label_config}'，全部填充为空")
            labels = [''] * len(tweets)
        else:
            labels = valid_df[label_config].fillna('').astype(str).tolist()
    elif isinstance(label_config, dict):
        cols = label_config.get('columns', [])
        func = label_config.get('func', combine_labels_by_any)
        # 在原始 df 上计算标签（避免切片后索引失效）
        combined = func(df, cols)
        labels = combined[non_empty].fillna('').astype(str).tolist()
    else:
        raise TypeError(f"label_config 类型错误: {type(label_config)}")

    print(f"  提取到 {len(tweets)} 条推文，有效标签数: {len([l for l in labels if l != ''])}")
    return tweets, labels

def main():
    all_tweets = []
    all_labels = []

    for idx, config in enumerate(file_configs, 1):
        print(f"\n处理文件 {idx}: {config['path']}")
        df = read_file_with_fallback(config)
        tweets, labels = extract_tweets_and_labels(
            df,
            config["tweet_column"],
            config.get("label_config")  # 注意这里用的是 label_config
        )
        all_tweets.extend(tweets)
        all_labels.extend(labels)

    # 保存原始合并数据
    raw_df = pd.DataFrame({'tweet': all_tweets, 'label': all_labels})
    raw_output = "all_datasets_with_labels.csv"
    raw_df.to_csv(raw_output, index=False, encoding='utf-8', quoting=csv.QUOTE_MINIMAL)
    print(f"\n✓ 原始合并: {len(raw_df)} 条记录，已保存至 {raw_output}")

    # 去重（基于 tweet）
    dedup_df = raw_df.drop_duplicates(subset=['tweet'], keep=DEDUP_KEEP)
    removed = len(raw_df) - len(dedup_df)
    print(f"✓ 去重后: {len(dedup_df)} 条记录（移除 {removed} 条重复）")

    dedup_output = "all_datasets_with_labels_dedup.csv"
    dedup_df.to_csv(dedup_output, index=False, encoding='utf-8', quoting=csv.QUOTE_MINIMAL)
    print(f"✓ 去重结果已保存至 {dedup_output}")
    print("\n去重后前5条预览：")
    print(dedup_df.head())

if __name__ == "__main__":
    main()


处理文件 1: es_hf_102024.csv
  ✓ 使用编码 utf-8 成功读取 es_hf_102024.csv
  提取到 29855 条推文，有效标签数: 29855

处理文件 2: es.csv
  ✓ 使用编码 utf-8 成功读取 es.csv
  提取到 24110 条推文，有效标签数: 24110

✓ 原始合并: 53965 条记录，已保存至 all_datasets_with_labels.csv
✓ 去重后: 50787 条记录（移除 3178 条重复）
✓ 去重结果已保存至 all_datasets_with_labels_dedup.csv

去重后前5条预览：
                                               tweet label
0  Eran tan pero tan feministas que invisibilizab...   0.0
1  @USER @USER @USER Me carga en lo q se convirti...   0.0
2  , ¿Sabrán las femiorcas como @USER y todo el f...   1.0
3  @USER @USER @USER @USER Una vecina que nada te...   0.0
4   @USER Debajo de que piedra estaba ese flaiterio?   0.0


In [1]:
import pandas as pd

# 读取 parquet 文件
df = pd.read_parquet("zh-00000-of-00001.parquet", engine="pyarrow")  # 或 engine="fastparquet"

# 保存为 CSV
df.to_csv("zh-00000-of-00001.csv", index=False)

print("转换完成，已保存为 zh-00000-of-00001.csv")

转换完成，已保存为 zh-00000-of-00001.csv
